# 0. Preparation

## 0a. Imports

In [1]:
import os
import glob
import shutil
import zipfile
from tqdm import tqdm

import numpy as np
import pandas as pd
from typing import Union
from collections import Counter

import ruptures as rpt
from kneed import KneeLocator

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import splprep, splev
from scipy.signal import butter, filtfilt, savgol_filter

from RecordMuse.convert import convert as con
from RecordMuse.processing import filter as fil
from RecordMuse.processing import normalize as norm
from RecordMuse.analysis import psd

import importlib
importlib.reload(con)
importlib.reload(fil)
importlib.reload(norm)
importlib.reload(psd)

<module 'RecordMuse.analysis.psd' from '/Users/ryankim/Documents/CodeProjects/_RESEARCH/pedestrian_encounters/RecordMuse/analysis/psd.py'>

## 0b. Global Variables

In [3]:
_participants = [
    'P1', 'P2', 'P3', 'P4', 
    'P5', 'P6', 'P7', 'P8', 
    'P9', 'P10', 'P13', 
    'PI1', 'PI2'
]

_valid_participants = [
    p for p in _participants if p not in ['P3','PI1']
]

_trial_names = [
    "Trial-ApproachAudio Start", 
    "Trial-BehindAudio Start", 
    "Trial-Behind Start", 
    "Trial-AlleyRunnerAudio Start", 
    "Trial-AlleyRunner Start", 
    "Trial-Approach Start"
]

_trial_target_positions = [
    (6.78, -2),     # 0th trial: calibration
    (-6.78, -2),    # 1st trial at index 1
    (6.78, -2),     # 2nd trial
    (-6.78, -2),    # 3rd trial
    (6.78, -2),     # 4th trial
    (-6.78, -2),    # 5th trial
    (6.78, -2),     # 6th trial
    (-6.78, -2),    # 7th trial: extra
]

_trial_goal_axes = [
    np.arctan2(0,1),    # 0th trial: calibration
    np.arctan2(0,-1),   # 1st trial at index 1
    np.arctan2(0,1),    # 2nd trial
    np.arctan2(0,-1),   # 3rd trial
    np.arctan2(0,1),    # 4th trial
    np.arctan2(0,-1),   # 5th trial
    np.arctan2(0,1),    # 6th trial
    np.arctan2(0,-1),   # 7th tria,: extra 
]

# 1. Extracting Timestamps

In [4]:
"""
Given the eye data (or some file with trial-length timestamps and frames), gnerate a new DataFrame purely containing timestamps.
Returns: The timestamp dataframe
"""
def timestamps(
    filepath:str,
    ts_colname:str = 'unix_ms',
    frame_colname:str = 'frame',
    outpath:str = None
):
    df = pd.read_csv(filepath)
    df = df[[ts_colname, frame_colname]]
    if outpath is not None:
        outdir = os.path.split(outpath)[0]
        os.makedirs(outdir, exist_ok=True)
        df.to_csv(outpath, index=False)
    return df

# 2. Handle Calibrations